In [ ]:
'''
Run with the following to be able to import from 'core_experiments' & `remoteobj` natively
`uv run --no-project --python 3.12 --with risq/software/arm --with risq/software/common/remoteobj/ --with ipython ipython`

Alternatevly use whatever environment manager you prefer
'''

In [ ]:
'''
The following demonstrates how to connect to several boards with remoteobj
'''
UNDEFINED = "UNDEFINED"
from os import getenv; from pathlib import Path; from remoteobj.client import Proxy
RISQ_HOST, RISQ_PORTS = getenv("RISQ_HOST", UNDEFINED), [int(port) for port in getenv("RISQ_PORTS", UNDEFINED).split(",")]
RISQ_ROOT, RISQ_PROJECT =Path(getenv("RISQ_ROOT", UNDEFINED)),"risq"

if RISQ_HOST != UNDEFINED:
    if RISQ_PORTS == [UNDEFINED]:
        sessions = [Proxy.to_host(host=RISQ_HOST)]
    else:
        sessions = [Proxy.to_host(host=RISQ_HOST, port=port) for port in RISQ_PORTS]
    RISQS = [session.RISQ.__call__ for session in sessions]
else:
    raise RuntimeError(
        "No target board defined to attach to." \
        "Please define a RISQ_HOST in your environment with the IP of your board."
        )
risqs = [RISQ(project=RISQ_PROJECT,
              xsa=Path("build", "risq.xsa"),
              dtbo=Path("build", "risq.dtbo"))
              for RISQ in RISQS]

In [8]:
'''
For this demo I'll just need a single board
'''
from core_experiments import CoreExperiments
experiment = CoreExperiments(risqs[0])
experiment.run_orc(log2_shots=10, phase_offset=0, graph=False)
experiment.run_t1(rounds=100, time_steps=500, time_increment_in_cycles=500, shots=10)
experiment.run_rabi(rounds=100, initial_gain=-25000, gain_steps=500, gain_increment=100, shots=10)
experiment.run_ramsey(rounds=100, time_steps=500, time_increment_in_cycles=125, phase_step=0, shots=10)

NameError: name 'risqs' is not defined

In [ ]:
from dwave.elm.interface.pulse_definitions import PulseDefinitions
from pprint import pprint
pulse_definitions = PulseDefinitions()

system_names = [str(system.system.name) for system in pulse_definitions.elm_environment_state.system_versions]
print(f"{system_names=}\n\n")

for system_name in system_names:
    system = pulse_definitions.get_system_by_name(pulse_definitions.elm_environment_state, system_name)
    print(f"{system=}\n")

    parameter_roles = pulse_definitions.role_labels(pulse_definitions.registry, system)
    print(f"{parameter_roles=}\n")

    if parameters := pulse_definitions.get_parameters_by_role(system, "pulse_definitions"):
        all_pulse_data = pulse_definitions.get_pulse_data(system)
        print(f"{all_pulse_data=}\n")

        for pulse_name, pulse_data in all_pulse_data.items():
            pulse_parameters = pulse_data["parameters"]
            print(f"Pulse: {pulse_name}\n  Parameters: {pulse_parameters=}")

        quantum_frequency = pulse_definitions.get_quantum_frequency(system)
        print(f"{quantum_frequency=}")

    print("-"*20)


system_names=['q0.mem_readout', 'card1_q0.pair1', 'q0.mem_transmon', 'card1_q0.configuration']


system=System(uuid=UUID('1cbd578b-818b-4c5f-893d-c05c7886e7f9'), name='q0.mem_readout', kind_ref=SystemKindRef(uuid=UUID('2de8ea74-395c-4897-b6a0-e3e7abf244d5'), version_number=1), committed_at=None, attributes=[EntityAttribute(role_name='mode_id', values=[IntValue(value_type=<ValueType.INT: 'int'>, int_value=3)])], parameters=[EntityParameter(role_name='quantum_frequency', values=[FloatValue(value_type=<ValueType.FLOAT: 'float'>, float_value=1500000000.0)]), EntityParameter(role_name='aithex_ssb', values=[FloatValue(value_type=<ValueType.FLOAT: 'float'>, float_value=50000000.0)]), EntityParameter(role_name='log_true_acquisition_length', values=[IntValue(value_type=<ValueType.INT: 'int'>, int_value=2500)]), EntityParameter(role_name='log_true_acquisition_delay', values=[IntValue(value_type=<ValueType.INT: 'int'>, int_value=300)]), EntityParameter(role_name='pulse_definitions', values=[Objec

KeyError: "System 'card1_q0.pair1' has no quantum_frequency value"

In [ ]:

# get transmon system
# transmon_system_name = "q0.mem_transmon"
# transmon_system = pulse_definitions.get_system_by_name(pulse_definitions.elm_environment_state, transmon_system_name)
# print(f"Selected transmon system: {transmon_system.name}")

# # get parameters on transmon system
# transmon_parameter_roles = pulse_definitions.role_labels(pulse_definitions.registry, transmon_system)
# print(f"{transmon_parameter_roles=}")
# print()

# all_pulse_data = pulse_definitions.get_pulse_data(transmon_system)

# constant_pulse = all_pulse_data["constant"]
# if not isinstance(constant_pulse, dict):
#     raise TypeError("constant pulse must be an object")
# constant_pulse_parameters = constant_pulse["parameters"]
# if not isinstance(constant_pulse_parameters, dict):
#     raise TypeError("constant pulse parameters must be an object")
# constant_pulse_length = constant_pulse_parameters["length"]

# pi_pulse = all_pulse_data["pi"]
# if not isinstance(pi_pulse, dict):
#     raise TypeError("pi pulse must be an object")
# pi_pulse_parameters = pi_pulse["parameters"]
# if not isinstance(pi_pulse_parameters, dict):
#     raise TypeError("pi pulse parameters must be an object")
# pi_pulse_offset = pi_pulse_parameters["offset"]

quantum_frequency = pulse_definitions.get_quantum_frequency(transmon_system)

pprint(f"pulse_types: {list(all_pulse_data)}")
print()
pprint(
    {
        "constant_pulse": constant_pulse,
        "constant_pulse_length": constant_pulse_length,
        "pi_pulse": pi_pulse,
        "pi_pulse_offset": pi_pulse_offset,
        **pulse_definitions.with_units(
            pulse_definitions.registry,
            transmon_system,
            {
                "quantum_frequency": quantum_frequency,
                **pulse_definitions.get_transmon_calibration(transmon_system),
            },
        ),
    }
)

# get readout system
readout_system_name = "q0.mem_readout"
readout_system = pulse_definitions.get_system_by_name(pulse_definitions.elm_environment_state, readout_system_name)
print(f"Selected readout system: {readout_system.name}")
print()

print("Inspecting readout system")
print(readout_system)

# get parameters on readout system
readout_parameter_roles = pulse_definitions.role_labels(pulse_definitions.registry, readout_system)
print(f"{readout_parameter_roles=}")
print()

all_pulse_data = pulse_definitions.get_pulse_data(readout_system)

constant_pulse = all_pulse_data["constant"]
if not isinstance(constant_pulse, dict):
    raise TypeError("constant pulse must be an object")
constant_pulse_parameters = constant_pulse["parameters"]
if not isinstance(constant_pulse_parameters, dict):
    raise TypeError("constant pulse parameters must be an object")
constant_pulse_length = constant_pulse_parameters["length"]

gaussian_flattop_pulse = all_pulse_data["gaussian_flattop"]
if not isinstance(gaussian_flattop_pulse, dict):
    raise TypeError("gaussian_flattop pulse must be an object")
gaussian_flattop_pulse_parameters = gaussian_flattop_pulse["parameters"]
if not isinstance(gaussian_flattop_pulse_parameters, dict):
    raise TypeError("gaussian_flattop pulse parameters must be an object")
gaussian_flattop_pulse_rise_time = gaussian_flattop_pulse_parameters["rise_time"]

quantum_frequency = pulse_definitions.get_quantum_frequency(readout_system)

pprint(f"pulse_types: {list(all_pulse_data)}")
print()
pprint(
    {
        "constant_pulse": constant_pulse,
        "constant_pulse_length": constant_pulse_length,
        "gaussian_flattop_pulse": gaussian_flattop_pulse,
        "gaussian_flattop_pulse_rise_time": gaussian_flattop_pulse_rise_time,
        **pulse_definitions.with_units(
            pulse_definitions.registry,
            readout_system,
            {
                "quantum_frequency": quantum_frequency,
                "aithex_ssb": pulse_definitions.get_float_parameter(readout_system, "aithex_ssb"),
                **pulse_definitions.get_readout_acquisition(readout_system),
            },
        ),
    }
)

# inspect card data
card1_q0_pair1 = pulse_definitions.get_system_by_name(
    environment_state=pulse_definitions.elm_environment_state, name="card1_q0.pair1"
)

card1_q0_config = pulse_definitions.get_system_by_name(
    environment_state=pulse_definitions.elm_environment_state, name="card1_q0.configuration"
)

pprint(
    pulse_definitions.with_units(
        pulse_definitions.registry,
        card1_q0_config,
        {"mixer_freq_dac0": pulse_definitions.get_mixer_freq_dac0(card1_q0_config)},
    )
)